# Venue Effect — Preprocessing: Build Treated & Control DataFrames

**Goal:** For each field (physics, biology, chemistry, sociology):
- **Treated**: authors who published in any of the selected top journals. Only their **first** publication year counts as the treatment event.
- **Control**: all authors in the field (by modal ANZSRC code) who have **never** published in any of those journals, filtered to career_age ≥ 15.

**Design choices:**
- All original columns are kept — column selection happens at matching time.
- Country code columns (REPEATED arrays from BQ) stay as lists → use set-overlap for matching later.
- No journal hierarchy / tier ranking. All selected journals are treated equally.

**Input:**
- `../../data/author_info/` — author_panel_complete parquet shards (exported from BQ, files may lack .parquet extension)
- `../../data/top_venue_author/{field}/` — top_venue_{field}_modal parquet shards (exported from BQ)

**Output** (one treated + one control per field):
- `../../data/matching_needed/treated_df_{field}.parquet` — all treated authors, has `journal_id` column for filtering at matching time
- `../../data/matching_needed/control_df_{field}.parquet` — all control authors (never published in any listed journal)

In [15]:
import os
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## 1. Configuration

In [16]:
DATA_DIR = Path('../../data')
OUTPUT_DIR = DATA_DIR / 'matching_needed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Which fields to process (comment out any you want to skip)
FIELDS_TO_PROCESS = ['physics', 'biology', 'chemistry', 'sociology']

In [17]:
# Journal IDs from BigQuery (dimensions-ai.data_analytics.publications)
# Matched exactly to the SQL in Step 4 of the pipeline.
#
# Treated = authors from ANY of these journals.
# Control = field authors who NEVER published in any of these.

FIELD_CONFIG = {
    'physics': {
        'modal_field_code': '51',   # ANZSRC: Physical Sciences
        'journals': {
            'jour.1018957': 'Nature',
            'jour.1346339': 'Science',
            'jour.1082971': 'PNAS',
            'jour.1034717': 'Nature Physics',
            'jour.1018277': 'Physical Review Letters',   # PRL
            'jour.1053349': 'Physical Review A',         # PRA
            'jour.1320488': 'Physical Review B',         # PRB
            'jour.1320490': 'Physical Review C',         # PRC
            'jour.1320496': 'Physical Review D',         # PRD
            'jour.1312290': 'Physical Review E',         # PRE
        },
    },
    'biology': {
        'modal_field_code': '31',   # ANZSRC: Biological Sciences
        'journals': {
            'jour.1018957': 'Nature',
            'jour.1346339': 'Science',
            'jour.1082971': 'PNAS',
            'jour.1019114': 'Cell',
            'jour.1103138': 'Nature Genetics',
            'jour.1021344': 'Nature Cell Biology',
            'jour.1295033': 'Nature Structural & Molecular Biology',
        },
    },
    'chemistry': {
        'modal_field_code': '34',   # ANZSRC: Chemical Sciences
        'journals': {
            'jour.1018957': 'Nature',
            'jour.1346339': 'Science',
            'jour.1082971': 'PNAS',
            'jour.1081898': 'Journal of the American Chemical Society',
            'jour.1017044': 'Angewandte Chemie International Edition',
            'jour.1041224': 'Nature Chemistry',
            'jour.1155085': 'Chem',
        },
    },
    'sociology': {
        'modal_field_code': '44',   # ANZSRC: Human Society
        'journals': {
            'jour.1013002': 'AJS',
            'jour.1017026': 'American Sociological Review',
            'jour.1027842': 'European Sociological Review',
            'jour.1068714': 'Social Forces',
            'jour.1126009': 'Gender & Society',
            'jour.1008496': 'Sociology',
        },
    },
}

# Flat lookup: journal_id → short label
JOURNAL_ID_TO_LABEL = {}
for _cfg in FIELD_CONFIG.values():
    JOURNAL_ID_TO_LABEL.update(_cfg['journals'])

print(f'Total journal IDs across all fields: {len(JOURNAL_ID_TO_LABEL)}')

Total journal IDs across all fields: 24


## 2. Helper Functions

In [18]:
def read_parquet_dir(directory):
    """
    Read all parquet shards from a directory and return a DataFrame.
    Handles BQ EXPORT DATA files (may lack .parquet extension).
    
    Uses pyarrow.dataset for speed — reads all shards in parallel
    instead of one-by-one pd.read_parquet.
    """
    import pyarrow.dataset as ds
    import time
    
    t0 = time.time()
    dataset = ds.dataset(directory, format='parquet')
    df = dataset.to_table().to_pandas()
    elapsed = time.time() - t0
    
    print(f'  Read {directory}')
    print(f'  → {len(df):,} rows, {len(df.columns)} cols  ({elapsed:.1f}s)')
    return df


def assign_career_stage(career_age):
    """
    career_age = first_publish_year_in_top_venue - first_year_of_publication
      early:  <= 15
      mid:    (15, 40]
      late:   > 40
    """
    return np.select(
        [career_age <= 15,
         (career_age > 15) & (career_age <= 40),
         career_age > 40],
        ['early-career', 'mid-career', 'late-career'],
        default='unknown'
    )

## 3. Load Author Panel (once, shared across fields)

In [19]:
author_panel = read_parquet_dir(str(DATA_DIR / 'author_info/author_panel_complete'))

print(f'\nTotal authors: {author_panel["author_id"].nunique():,}')
print(f'Year range: {author_panel["year"].min()} – {author_panel["year"].max()}')

  Read ../../data/author_info/author_panel_complete
  → 132,838,357 rows, 35 cols  (115.9s)

Total authors: 6,670,317
Year range: 1666 – 2023


In [20]:
# Column inventory
print(f'Columns ({len(author_panel.columns)}):')
for c in author_panel.columns:
    print(f'  {c:<45s}  {author_panel[c].dtype}')

Columns (35):
  author_id                                      object
  year                                           int64
  author_first_name                              object
  author_last_name                               object
  first_year_of_publication                      int64
  first_affiliation_countries                    object
  first_ever_affiliation_countries               object
  modal_field_code                               object
  modal_field_fraction                           float64
  career_age                                     int64
  publication_count                              int64
  corresponding_author_count                     int64
  citations_annual                               int64
  funding_count                                  int64
  average_funding                                float64
  amount_funding                                 float64
  unique_coauthors_count                         int64
  collaborations_count                 

In [21]:
# Detect list/array columns (REPEATED fields from BQ)
list_cols = []
for c in author_panel.columns:
    if author_panel[c].dtype == object:
        sample = author_panel[c].dropna().head(100)
        if sample.apply(type).eq(list).any():
            list_cols.append(c)

if list_cols:
    print(f'Array/list columns: {list_cols}')
    for c in list_cols:
        val = author_panel[c].dropna().iloc[0] if author_panel[c].notna().any() else None
        print(f'  {c}: e.g. {val}')
    print('→ Kept as-is. Use set-overlap for matching later.')
else:
    print('No list/array columns detected.')

No list/array columns detected.


In [22]:
# Modal field distribution
author_panel.groupby('modal_field_code')['author_id'].nunique() \
    .sort_values(ascending=False).head(10)

modal_field_code
31    2889699
34    2114572
51    1090349
44     575697
Name: author_id, dtype: int64

## 4. Core Processing Functions

In [23]:
def get_first_venue_publication(top_venue_df):
    """
    For each author in top_venue_df, find their FIRST publication year
    across all selected journals.
    
    An author who published 5 times in Science only gets ONE treatment
    event: the earliest year.
    
    Returns one row per author:
      author_id, first_publish_year, journal_id, journal_title, journal_label
    """
    tv = top_venue_df.copy()
    tv['journal_label'] = tv['journal_id'].map(JOURNAL_ID_TO_LABEL)
    
    # Earliest year per author
    first_year = tv.groupby('author_id')['year'].min().rename('first_publish_year')
    tv = tv.merge(first_year, on='author_id')
    
    # Keep only rows from that first year
    tv = tv[tv['year'] == tv['first_publish_year']]
    
    # One row per author (deterministic tie-break by journal_id)
    tv = (tv.sort_values(['author_id', 'journal_id'])
            .drop_duplicates(subset='author_id', keep='first'))
    
    return tv[['author_id', 'first_publish_year',
               'journal_id', 'journal_title', 'journal_label']].copy()

In [24]:
def build_treated_df(field_panel, top_venue_df, field_name):
    """
    Treated = authors from top_venue_df, merged with their full
    author_panel history.
    
    Keeps ALL original panel columns. Adds:
      first_publish_year, journal_id, journal_title, journal_label,
      to_year, career_age_at_treatment, new_career_stage
    """
    print(f'\n--- Building treated_df for {field_name} ---')
    
    # One row per author: their first venue publication
    first_pubs = get_first_venue_publication(top_venue_df)
    print(f'  Unique treated authors (in venue data): {len(first_pubs):,}')
    print(f'  By journal:')
    print(first_pubs['journal_label'].value_counts().to_string())
    
    # Inner join with author panel → keeps ALL panel columns
    treated = field_panel.merge(first_pubs, on='author_id', how='inner')
    print(f'  After panel merge: {treated["author_id"].nunique():,} authors, '
          f'{len(treated):,} rows')
    
    # Derived columns
    treated['to_year'] = treated['year'] - treated['first_publish_year']
    treated['career_age_at_treatment'] = (
        treated['first_publish_year'] - treated['first_year_of_publication']
    )
    treated['new_career_stage'] = assign_career_stage(
        treated['career_age_at_treatment']
    )
    
    return treated

In [25]:
def build_control_df(full_author_panel, treated_author_ids,
                     field_code, field_name):
    """
    Control = all authors in the field who NEVER appear in the treated set,
    filtered to career_age >= 15.
    
    Keeps ALL original panel columns.
    """
    print(f'\n--- Building control_df for {field_name} ---')
    
    # Filter to field
    ctrl = full_author_panel[
        full_author_panel['modal_field_code'] == field_code
    ].copy()
    n_field = ctrl['author_id'].nunique()
    print(f'  Total authors in field {field_code}: {n_field:,}')
    
    # STRICTLY exclude treated authors
    ctrl = ctrl[~ctrl['author_id'].isin(treated_author_ids)]
    n_after = ctrl['author_id'].nunique()
    print(f'  After excluding {n_field - n_after:,} treated: {n_after:,}')
    
    # Ensure career_age column exists
    if 'career_age' not in ctrl.columns:
        ctrl['career_age'] = ctrl['year'] - ctrl['first_year_of_publication']
    
    # Keep authors who reach career_age >= 15 at some point
    valid_ids = ctrl.loc[ctrl['career_age'] >= 15, 'author_id'].unique()
    ctrl = ctrl[ctrl['author_id'].isin(valid_ids)]
    print(f'  After career_age >= 15 filter: {ctrl["author_id"].nunique():,} authors, '
          f'{len(ctrl):,} rows')
    
    return ctrl

## 5. Process All Fields

For each field:
1. Load the top venue parquet
2. Build treated_df (first pub in any selected journal + full panel)
3. Build control_df (field authors who never published in those journals)
4. Verify zero overlap
5. Save

In [27]:
for field_name in FIELDS_TO_PROCESS:
    print('=' * 70)
    print(f'FIELD: {field_name.upper()}')
    print('=' * 70)
    
    config = FIELD_CONFIG[field_name]
    field_code = config['modal_field_code']
    
    # --- Load top venue data ---
    venue_dir = DATA_DIR / 'top_venue_author' / field_name
    if not venue_dir.exists():
        print(f'  WARNING: {venue_dir} not found, skipping')
        continue
    
    top_venue_df = read_parquet_dir(str(venue_dir))
    print(f'  Venue pubs: {len(top_venue_df):,}')
    print(f'  Venue authors: {top_venue_df["author_id"].nunique():,}')
    
    # --- Field panel ---
    field_panel = author_panel[
        author_panel['modal_field_code'] == field_code
    ].copy()
    print(f'  Field panel authors (modal={field_code}): '
          f'{field_panel["author_id"].nunique():,}')
    
    # --- Treated ---
    treated_df = build_treated_df(field_panel, top_venue_df, field_name)
    treated_ids = set(treated_df['author_id'].unique())
    
    # --- Control ---
    control_df = build_control_df(author_panel, treated_ids,
                                  field_code, field_name)
    
    # --- Sanity check: ZERO overlap ---
    overlap = treated_ids & set(control_df['author_id'].unique())
    assert len(overlap) == 0, \
        f'BUG: {len(overlap)} authors in both treated and control!'
    print(f'\n  ✓ Overlap check: 0 authors in both sets')
    
    # --- Summary ---
    print(f'\n  SUMMARY')
    print(f'  Treated: {treated_df["author_id"].nunique():,} authors, '
          f'{len(treated_df):,} rows')
    print(f'  Control: {control_df["author_id"].nunique():,} authors, '
          f'{len(control_df):,} rows')
    
    fp = treated_df.drop_duplicates(subset='author_id')
    print(f'\n  Treated by journal:')
    print(fp['journal_label'].value_counts().to_string())
    print(f'\n  Treated career stage:')
    print(fp['new_career_stage'].value_counts().to_string())
    if 'Gender' in fp.columns:
        print(f'\n  Treated gender (0=unk, 1=M, 2=F):')
        print(fp['Gender'].value_counts().sort_index().to_string())
    
    # --- Save ---
    t_path = OUTPUT_DIR / f'treated_df_{field_name}.parquet'
    c_path = OUTPUT_DIR / f'control_df_{field_name}.parquet'
    treated_df.to_parquet(t_path, index=False)
    control_df.to_parquet(c_path, index=False)
    print(f'\n  Saved: {t_path.name}, {c_path.name}')
    
    print()

FIELD: PHYSICS
  Read ../../data/top_venue_author/physics
  → 3,871,685 rows, 13 cols  (4.8s)
  Venue pubs: 3,871,685
  Venue authors: 335,730
  Field panel authors (modal=51): 1,090,349

--- Building treated_df for physics ---
  Unique treated authors (in venue data): 335,730
  By journal:
journal_label
Physical Review Letters    101159
Physical Review B           78293
Physical Review D           41444
Physical Review A           36282
Physical Review C           25887
Nature                      20929
Physical Review E           14508
Science                      8515
Nature Physics               6561
PNAS                         2152
  After panel merge: 335,580 authors, 8,038,079 rows

--- Building control_df for physics ---
  Total authors in field 51: 1,090,349
  After excluding 335,580 treated: 754,769
  After career_age >= 15 filter: 372,574 authors, 12,938,268 rows

  ✓ Overlap check: 0 authors in both sets

  SUMMARY
  Treated: 335,580 authors, 8,038,079 rows
  Control: 372,

## 6. Quick Verification

Load back one field and check.

In [30]:
# Pick a field to verify
verify_field = 'physics'

t = pd.read_parquet(OUTPUT_DIR / f'treated_df_{verify_field}.parquet')
c = pd.read_parquet(OUTPUT_DIR / f'control_df_{verify_field}.parquet')

print(f'{verify_field} treated: {t["author_id"].nunique():,} authors, {len(t):,} rows')
print(f'{verify_field} control: {c["author_id"].nunique():,} authors, {len(c):,} rows')

# Zero overlap
overlap = set(t['author_id']) & set(c['author_id'])
print(f'Overlap: {len(overlap)} (should be 0)')

# Each treated author appears once per year (panel structure)
dupes = t.groupby(['author_id', 'year']).size()
print(f'Max rows per (author, year): {dupes.max()} (should be 1)')

# Each treated author has exactly one first_publish_year
fpyrs = t.groupby('author_id')['first_publish_year'].nunique()
print(f'Max distinct first_publish_year per author: {fpyrs.max()} (should be 1)')

physics treated: 335,580 authors, 8,038,079 rows
physics control: 372,574 authors, 12,938,268 rows
Overlap: 0 (should be 0)
Max rows per (author, year): 2 (should be 1)
Max distinct first_publish_year per author: 1 (should be 1)


In [31]:
# Peek at treated
t.head(3)

,author_id,year,author_first_name,author_last_name,first_year_of_publication,first_affiliation_countries,first_ever_affiliation_countries,modal_field_code,modal_field_fraction,career_age,publication_count,corresponding_author_count,citations_annual,funding_count,average_funding,amount_funding,unique_coauthors_count,collaborations_count,is_first_author,is_last_author,avg_normalized_citations,sum_normalized_citations,normalized_productivity,normalized_funding_count,current_year_affiliation_countries,cum_publication_count,cum_corresponding_count,cum_citations,cum_funding_count,cum_amount_funding,cum_collaborations_count,cum_normalized_citations,cum_normalized_productivity,cum_normalized_funding_count,Gender,first_publish_year,journal_id,journal_title,journal_label,to_year,career_age_at_treatment,new_career_stage
0,ur.01000001015.36,1988,A,Talneau,1988,[FR],[FR],51,0.4868,0,1,0,0,0,0.0,0.0,3,3,1,0,0.2777,0.2777,1.6434,0.0,[FR],1,0,0,0,0.0,3,0.2777,1.6434,0.0,1.0,2004,jour.1018277,Physical Review Letters,Physical Review Letters,-16,16,mid-career
1,ur.01000001015.36,1989,A,Talneau,1988,[FR],[FR],51,0.4868,1,1,0,1,0,0.0,0.0,7,7,0,0,0.5237,0.5237,1.6656,0.0,[FR],2,0,1,0,0.0,10,0.8014,3.3090,0.0,1.0,2004,jour.1018277,Physical Review Letters,Physical Review Letters,-15,16,mid-career
2,ur.01000001015.36,1990,A,Talneau,1988,[FR],[FR],51,0.4868,2,0,0,3,0,0.0,0.0,0,0,0,0,0.0000,0.0000,0.0000,0.0,[],2,0,4,0,0.0,10,0.8014,3.3090,0.0,1.0,2004,jour.1018277,Physical Review Letters,Physical Review Letters,-14,16,mid-career


In [36]:
t.value_counts('Gender')

Gender
0.0    4543430
1.0     854078
Name: count, dtype: int64

In [32]:
# Peek at control
c.head(3)

,author_id,year,author_first_name,author_last_name,first_year_of_publication,first_affiliation_countries,first_ever_affiliation_countries,modal_field_code,modal_field_fraction,career_age,publication_count,corresponding_author_count,citations_annual,funding_count,average_funding,amount_funding,unique_coauthors_count,collaborations_count,is_first_author,is_last_author,avg_normalized_citations,sum_normalized_citations,normalized_productivity,normalized_funding_count,current_year_affiliation_countries,cum_publication_count,cum_corresponding_count,cum_citations,cum_funding_count,cum_amount_funding,cum_collaborations_count,cum_normalized_citations,cum_normalized_productivity,cum_normalized_funding_count,Gender
0,ur.01000000017.89,2006,C. A. J.,Norvill,2006,[AU],[AU],51,0.6,0,1,1,0,0,0.0,0.0,0,0,1,0,0.8389,0.8389,1.9066,0.0,[AU],1,1,0,0,0.0,0,0.8389,1.9066,0.0,0.0
1,ur.01000000017.89,2007,C. A. J.,Norvill,2006,[AU],[AU],51,0.6,1,0,0,0,0,0.0,0.0,0,0,0,0,0.0000,0.0000,0.0000,0.0,[],1,1,0,0,0.0,0,0.8389,1.9066,0.0,0.0
2,ur.01000000017.89,2008,C. A. J.,Norvill,2006,[AU],[AU],51,0.6,2,1,1,2,0,0.0,0.0,1,1,1,0,0.0489,0.0489,1.9897,0.0,[AU],2,2,2,0,0.0,1,0.8878,3.8963,0.0,0.0


In [33]:
# Column comparison
print('Treated columns:')
print(list(t.columns))
print(f'\nControl columns:')
print(list(c.columns))
print(f'\nExtra in treated (from venue merge): '
      f'{set(t.columns) - set(c.columns)}')

Treated columns:
['author_id', 'year', 'author_first_name', 'author_last_name', 'first_year_of_publication', 'first_affiliation_countries', 'first_ever_affiliation_countries', 'modal_field_code', 'modal_field_fraction', 'career_age', 'publication_count', 'corresponding_author_count', 'citations_annual', 'funding_count', 'average_funding', 'amount_funding', 'unique_coauthors_count', 'collaborations_count', 'is_first_author', 'is_last_author', 'avg_normalized_citations', 'sum_normalized_citations', 'normalized_productivity', 'normalized_funding_count', 'current_year_affiliation_countries', 'cum_publication_count', 'cum_corresponding_count', 'cum_citations', 'cum_funding_count', 'cum_amount_funding', 'cum_collaborations_count', 'cum_normalized_citations', 'cum_normalized_productivity', 'cum_normalized_funding_count', 'Gender', 'first_publish_year', 'journal_id', 'journal_title', 'journal_label', 'to_year', 'career_age_at_treatment', 'new_career_stage']

Control columns:
['author_id', 'yea

In [34]:
print('Done.')

Done.
